# Grafico 05 - Distribucion de la superficie natural segun distancia al borde del AP

Este notebook reproduce en Google Colab las 2 imagenes de este grafico (boxplot general y por tipologia) y su tabla de soporte en Excel.

**Antes de correr las celdas de abajo**, ten a mano el archivo `naturalidad_data.json` (esta en la carpeta `codigo/` de este grafico, en tu computador o en tu repositorio de GitHub).

Corre las celdas en orden, de arriba hacia abajo.

## 1. Instalar paquetes

In [ ]:
!pip -q install numpy pandas matplotlib openpyxl


## 2. Subir el archivo de datos

In [ ]:
from google.colab import files
print("Sube aqui: naturalidad_data.json")
uploaded = files.upload()


## 3. Generar las 2 imagenes

In [ ]:
"""
Distribución de la superficie natural según distancia al borde del AP
=========================================================================

Gráfico corregido según comentarios de Vale (agosto-2026): limpieza de
diseño y terminología, misma lógica de "general + por tipología" que el
resto de los gráficos de este proyecto. Ver README.txt y METODOLOGIA.docx
de esta carpeta para el detalle completo.

QUÉ HACE ESTE SCRIPT
--------------------
Genera 2 gráficos de cajas (boxplot) que muestran cómo se DISTRIBUYE (no
solo el promedio) el % de superficie natural entre las AP, en cada
distancia desde el borde, comparando los años 2000 y 2024:

  1. boxplot_general.png         -> distribución de las 97 AP juntas
  2. boxplot_por_tipologia.png   -> lo mismo, en 3 paneles (PN/RN/MN)

Cada "caja" resume la distribución de esa distancia y año: la línea del
medio es la mediana, la caja va del percentil 25 al 75, los "bigotes" se
extienden hasta el resto de los datos sin contar valores atípicos, y los
puntos sueltos son esos valores atípicos (AP con un % muy distinto al
resto del grupo en esa distancia).

DISEÑO: solo título + nombres de ejes + el gráfico (más la leyenda de año,
que es parte del gráfico) -- sin subtítulo ni notas al pie sobre la
imagen. Terminología corregida: "superficie natural", no "cobertura
natural".

ARCHIVO DE ENTRADA (debe estar en esta misma carpeta `codigo/`)
------------------------------------------------------------------
  naturalidad_data.json   -> % de superficie natural por AP, año y
                              distancia (serie "anillos": franja aislada).

SALIDA (se guarda en ../imagenes/)
-----------------------------------
  boxplot_general.png
  boxplot_por_tipologia.png

Para correrlo: python3 boxplot_distancia.py
(requiere numpy, matplotlib -- instalar con: pip install numpy matplotlib)

===========================================================================
QUÉ CAMBIAR SI...                                                (resumen)
===========================================================================
  ...moviste este script a otra carpeta y naturalidad_data.json no está al
     lado -> variable NATURALIDAD_JSON_PATH, más abajo.
  ...quieres que las imágenes se guarden en otro lugar
     -> variable OUT_DIR, más abajo.
  ...cambian los años que se comparan (hoy: 2000 vs 2024)
     -> variables ANIO_INICIAL y ANIO_FINAL, más abajo.
  ...quieres cambiar tamaño de letra, colores, tamaño de figura, etc.
     (ajustes puramente visuales)
     -> están marcados con "<-- AJUSTE VISUAL" en cada sección.
===========================================================================
"""

import json
import re
import os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ---------------------------------------------------------------------
# RUTAS DE ARCHIVOS
# ---------------------------------------------------------------------
BASE_DIR = "/content"  # <-- en Colab, los archivos subidos con files.upload() quedan en /content

# <-- CAMBIAR AQUÍ si le cambiaste el nombre al archivo de datos, o si lo
#     moviste a otra carpeta.
NATURALIDAD_JSON_PATH = os.path.join(BASE_DIR, "naturalidad_data.json")

# <-- CAMBIAR AQUÍ si quieres que los PNG se guarden en otro lugar (por
#     defecto: una carpeta "imagenes" al lado de esta carpeta "codigo").
OUT_DIR = os.path.join(BASE_DIR, "imagenes")
os.makedirs(OUT_DIR, exist_ok=True)

# <-- CAMBIAR AQUÍ los 2 años que se comparan.
ANIO_INICIAL = 2000
ANIO_FINAL = 2024

# ------------------------------------------------------------------
# PALETA Y ESTILO (igual al resto del proyecto, para consistencia visual)
# ------------------------------------------------------------------
SURFACE = "#fcfcfb"
INK_PRIMARY = "#0b0b0b"
INK_SECONDARY = "#52514e"
INK_MUTED = "#898781"
GRID = "#e1e0d9"
BASELINE = "#c3c2b7"
CAT_BLUE = "#2a78d6"    # año inicial
CAT_ORANGE = "#eb6834"  # año final

plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Helvetica"]


def style_ax(ax):
    ax.set_facecolor(SURFACE)
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)
    for s in ["left", "bottom"]:
        ax.spines[s].set_color(BASELINE)
    ax.tick_params(colors=INK_MUTED, labelsize=9)
    ax.xaxis.label.set_color(INK_SECONDARY)
    ax.yaxis.label.set_color(INK_SECONDARY)


def style_box(bp, color):
    """Aplica el estilo visual estándar del proyecto a un boxplot de
    matplotlib (colores de caja, bigotes, mediana y valores atípicos)."""
    for box in bp["boxes"]:
        box.set(facecolor=color, edgecolor=INK_SECONDARY, linewidth=0.9, alpha=0.85)
    for el in ["whiskers", "caps"]:
        for line in bp[el]:
            line.set(color=INK_SECONDARY, linewidth=0.9)
    for med in bp["medians"]:
        med.set(color=INK_PRIMARY, linewidth=1.6)
    for fl in bp["fliers"]:
        fl.set(marker="o", markersize=3, markerfacecolor=color, markeredgecolor="none", alpha=0.6)
        # <-- AJUSTE VISUAL: markersize=3 (tamaño de los puntos de valores atípicos)


# ------------------------------------------------------------------
# 1. CARGA DE DATOS
# ------------------------------------------------------------------
DATA = json.load(open(NATURALIDAD_JSON_PATH))
DIST = DATA["dist"]
APS = DATA["aps"]
ANILLOS = DATA["anillos"]
x_order = DIST
n = len(x_order)


def tipologia(nombre):
    """PN/RN/MN a partir del prefijo del nombre de la AP.
    <-- CAMBIAR AQUÍ si el formato de los nombres cambia en el futuro."""
    m = re.match(r"^(MN|PN|RN)\s", nombre)
    return m.group(1) if m else "??"


TIPO_ORDER = ["PN", "RN", "MN"]  # <-- CAMBIAR AQUÍ si se agrega una 4ta tipología
TIPO_NOMBRE = {"PN": "Parque Nacional", "RN": "Reserva Nacional", "MN": "Monumento Natural"}

for a in APS:
    a["tipo"] = tipologia(a["name"])


def ap_val(ap_name, year, dist_label):
    idx = DIST.index(dist_label)
    return ANILLOS[ap_name][str(year)][idx]


def datos_por_distancia(nombres_ap):
    """Para una lista de nombres de AP, devuelve (data_inicial, data_final):
    2 listas de listas -- una lista de valores por cada distancia, para
    cada año -- listas para pasarle directo a ax.boxplot()."""
    data_inicial, data_final = [], []
    for d in x_order:
        v0 = [ap_val(nm, ANIO_INICIAL, d) for nm in nombres_ap]
        v1 = [ap_val(nm, ANIO_FINAL, d) for nm in nombres_ap]
        data_inicial.append([v for v in v0 if v is not None])
        data_final.append([v for v in v1 if v is not None])
    return data_inicial, data_final


# ------------------------------------------------------------------
# 2. GRÁFICO GENERAL (las 97 AP juntas)
# ------------------------------------------------------------------
data_2000, data_2024 = datos_por_distancia([a["name"] for a in APS])

fig, ax = plt.subplots(figsize=(8.4, 5.8), dpi=200)  # <-- AJUSTE VISUAL: tamaño y resolución de la imagen
fig.patch.set_facecolor(SURFACE)
style_ax(ax)
pos0 = np.arange(n) * 2.2  # <-- AJUSTE VISUAL: 2.2 (separación entre grupos de distancia)
pos1 = pos0 + 0.85          # <-- AJUSTE VISUAL: 0.85 (separación entre la caja del año inicial y la del año final)
bp0 = ax.boxplot(data_2000, positions=pos0, widths=0.7, patch_artist=True)
bp1 = ax.boxplot(data_2024, positions=pos1, widths=0.7, patch_artist=True)
style_box(bp0, CAT_BLUE)
style_box(bp1, CAT_ORANGE)
ax.set_xticks(pos0 + 0.425)
ax.set_xticklabels(x_order)
ax.set_ylabel(f"% superficie natural (distribución entre las {len(APS)} AP)")
ax.set_xlabel("Distancia desde el borde del AP")
ax.grid(axis="y", color=GRID, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
handles = [Patch(facecolor=CAT_BLUE, label=str(ANIO_INICIAL)), Patch(facecolor=CAT_ORANGE, label=str(ANIO_FINAL))]
leg = ax.legend(handles=handles, loc="lower left", frameon=False, fontsize=9)
for t in leg.get_texts():
    t.set_color(INK_SECONDARY)

# Solo título -- sin subtítulo/nota debajo (esa explicación va en el
# README/METODOLOGIA de esta carpeta).
fig.suptitle("Distribución de la superficie natural según distancia\nal borde del área protegida",
             color=INK_PRIMARY, fontsize=13.5, fontweight="bold", x=0.02, ha="left", y=0.995, va="top")
             # <-- AJUSTE VISUAL: fontsize=13.5 y posición (x, y) del título
fig.subplots_adjust(top=0.83, bottom=0.11, left=0.09, right=0.97)
fig.savefig(os.path.join(OUT_DIR, "boxplot_general.png"), facecolor=SURFACE)
plt.close(fig)
print("OK boxplot_general.png")

# ------------------------------------------------------------------
# 3. GRÁFICO POR TIPOLOGÍA (3 paneles: PN / RN / MN)
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 5.6), dpi=200, sharey=True)  # <-- AJUSTE VISUAL: tamaño de la imagen
fig.patch.set_facecolor(SURFACE)
for ax, t in zip(axes, TIPO_ORDER):
    style_ax(ax)
    aps_t = [a["name"] for a in APS if a["tipo"] == t]
    d0, d1 = datos_por_distancia(aps_t)
    p0 = np.arange(n) * 2.2
    p1 = p0 + 0.85
    bp0 = ax.boxplot(d0, positions=p0, widths=0.7, patch_artist=True)
    bp1 = ax.boxplot(d1, positions=p1, widths=0.7, patch_artist=True)
    style_box(bp0, CAT_BLUE)
    style_box(bp1, CAT_ORANGE)
    ax.set_xticks(p0 + 0.425)
    ax.set_xticklabels(x_order, fontsize=8, rotation=45, ha="right")
    ax.set_title(f"{TIPO_NOMBRE[t]} ({len(aps_t)} AP)", color=INK_PRIMARY, fontsize=11, fontweight="bold", pad=8)
    ax.set_xlabel("Distancia desde el borde del AP", fontsize=9)
    ax.grid(axis="y", color=GRID, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
axes[0].set_ylabel("% superficie natural (distribución)")
handles = [Patch(facecolor=CAT_BLUE, label=str(ANIO_INICIAL)), Patch(facecolor=CAT_ORANGE, label=str(ANIO_FINAL))]
leg = axes[-1].legend(handles=handles, loc="lower left", frameon=False, fontsize=9)
for t in leg.get_texts():
    t.set_color(INK_SECONDARY)

fig.suptitle("Distribución de la superficie natural según distancia,\npor tipología de área protegida", color=INK_PRIMARY,
             fontsize=14.5, fontweight="bold", x=0.02, ha="left", y=0.995, va="top")
             # <-- AJUSTE VISUAL: fontsize=14.5 y posición (x, y) del título
fig.subplots_adjust(top=0.80, bottom=0.2, left=0.05, right=0.98, wspace=0.08)
fig.savefig(os.path.join(OUT_DIR, "boxplot_por_tipologia.png"), facecolor=SURFACE)
plt.close(fig)
print("OK boxplot_por_tipologia.png")


## 4. Generar la tabla de soporte (Excel)

In [ ]:
"""
Tabla de soporte del gráfico 06 (distribución de superficie natural por distancia)
======================================================================================

Genera tabla_soporte.xlsx (en la carpeta de arriba, junto a README.txt) con
los datos exactos que resumen cada caja de los 2 boxplots, en 2 hojas:

  1. general_por_distancia     -> para las 97 AP juntas, en cada distancia
                                  y año: mínimo, percentil 25, mediana,
                                  percentil 75, máximo y cuántos valores
                                  atípicos ("fuera de rango") hay.
  2. por_tipologia_y_distancia -> lo mismo, calculado por separado dentro
                                  de cada tipología.

Nota: un boxplot resume una distribución completa (97 valores por celda),
no un solo número -- por eso esta tabla trae los 5 estadísticos que arma
cada caja, en vez de traer el dato crudo de las 97 AP (que ya está
disponible completo en la tabla de soporte del gráfico 01, hoja
heatmap_datos, si se necesita el detalle AP por AP).

Requiere: numpy, pandas, openpyxl
Para correrlo: python3 tabla_soporte.py

===========================================================================
QUÉ CAMBIAR SI...                                                (resumen)
===========================================================================
  ...moviste este script a otra carpeta y naturalidad_data.json no está al
     lado -> variable NATURALIDAD_JSON_PATH, más abajo.
  ...quieres que tabla_soporte.xlsx se guarde en otro lugar
     -> variable OUT_XLSX, más abajo.
  ...cambian los años que se comparan (hoy: 2000 vs 2024)
     -> variables ANIO_INICIAL y ANIO_FINAL, más abajo.
===========================================================================
"""

import json
import re
import os
import numpy as np
import pandas as pd

BASE_DIR = "/content"  # <-- en Colab, los archivos subidos con files.upload() quedan en /content
NATURALIDAD_JSON_PATH = os.path.join(BASE_DIR, "naturalidad_data.json")
OUT_XLSX = os.path.join(BASE_DIR, "tabla_soporte.xlsx")
ANIO_INICIAL = 2000  # <-- CAMBIAR AQUÍ si se comparan otros años
ANIO_FINAL = 2024

DATA = json.load(open(NATURALIDAD_JSON_PATH))
DIST = DATA["dist"]
APS = DATA["aps"]
ANILLOS = DATA["anillos"]


def tipologia(nombre):
    m = re.match(r"^(MN|PN|RN)\s", nombre)
    return m.group(1) if m else "??"


TIPO_ORDER = ["PN", "RN", "MN"]
for a in APS:
    a["tipo"] = tipologia(a["name"])


def ap_val(ap_name, year, dist_label):
    idx = DIST.index(dist_label)
    return ANILLOS[ap_name][str(year)][idx]


def resumen_boxplot(valores):
    """Los 5 estadísticos que dibuja un boxplot (estilo matplotlib, cuyos
    bigotes van hasta 1.5x el rango intercuartílico), más cuántos valores
    quedan fuera de ese rango (los puntos sueltos del gráfico)."""
    v = np.array([x for x in valores if x is not None], dtype=float)
    q1, mediana, q3 = np.percentile(v, [25, 50, 75])
    iqr = q3 - q1
    lo_fence, hi_fence = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    dentro = v[(v >= lo_fence) & (v <= hi_fence)]
    atipicos = v[(v < lo_fence) | (v > hi_fence)]
    return {
        "n": len(v), "minimo_bigote": round(float(dentro.min()), 2) if len(dentro) else None,
        "percentil_25": round(float(q1), 2), "mediana": round(float(mediana), 2),
        "percentil_75": round(float(q3), 2),
        "maximo_bigote": round(float(dentro.max()), 2) if len(dentro) else None,
        "n_valores_atipicos": int(len(atipicos)),
    }


def tabla_para(nombres_ap):
    rows = []
    for d in DIST:
        for anio, etiqueta in [(ANIO_INICIAL, "inicial"), (ANIO_FINAL, "final")]:
            vals = [ap_val(nm, anio, d) for nm in nombres_ap]
            r = resumen_boxplot(vals)
            r_ordenado = {"distancia": d, "anio": anio, "periodo": etiqueta, **r}
            rows.append(r_ordenado)
    return pd.DataFrame(rows)


# ---------------------------------------------------------------
# 1) HOJA general_por_distancia
# ---------------------------------------------------------------
df_general = tabla_para([a["name"] for a in APS])

# ---------------------------------------------------------------
# 2) HOJA por_tipologia_y_distancia
# ---------------------------------------------------------------
rows_tipo = []
for t in TIPO_ORDER:
    aps_t = [a["name"] for a in APS if a["tipo"] == t]
    df_t = tabla_para(aps_t)
    df_t.insert(0, "tipologia", t)
    rows_tipo.append(df_t)
df_tipo = pd.concat(rows_tipo, ignore_index=True)

with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    df_general.to_excel(writer, sheet_name="general_por_distancia", index=False)
    df_tipo.to_excel(writer, sheet_name="por_tipologia_y_distancia", index=False)

print(f"OK {OUT_XLSX} -- 2 hojas")


## 5. Ver las imagenes generadas

In [ ]:
import glob
from IPython.display import Image, display

for p in sorted(glob.glob(os.path.join(OUT_DIR, '*.png'))):
    print(p.split('/')[-1])
    display(Image(filename=p))


## 6. Descargar todo (imagenes + tabla de soporte) en un .zip

In [ ]:
import shutil, os
from google.colab import files

RESULT_DIR = "/content/resultados_05_boxplot"
os.makedirs(RESULT_DIR, exist_ok=True)
if os.path.isdir(OUT_DIR):
    shutil.copytree(OUT_DIR, os.path.join(RESULT_DIR, "imagenes"), dirs_exist_ok=True)
if os.path.exists(OUT_XLSX):
    shutil.copy(OUT_XLSX, RESULT_DIR)
shutil.make_archive(RESULT_DIR, "zip", RESULT_DIR)
files.download(RESULT_DIR + ".zip")
print("Listo:", RESULT_DIR + ".zip")
